In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_id = "perplexity-ai/pplx-embed-context-v1-4b"

# 1. Load Tokenizer & Model tối ưu cho GPU T4 Free Tier
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,   # Giảm dung lượng VRAM xuống 1/2
    device_map="auto",           # Đẩy thẳng lên GPU T4
    low_cpu_mem_usage=True       # Tránh tràn RAM hệ thống khi nạp mô hình
)

# 2. Hàm trích xuất Vector Embedding tối ưu bộ nhớ
def get_embedding(text):
    # Chuyển văn bản thành token và đẩy lên GPU
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cuda")

    with torch.no_grad(): # Tắt tính toán gradient để tiết kiệm VRAM và tăng tốc
        outputs = model(**inputs)

    # Mô hình PPLX-Embed trích xuất vector đại diện từ hidden state của token cuối cùng
    # (Hoặc sử dụng phương pháp Mean Pooling tùy thuộc vào tài liệu của Perplexity)
    embeddings = outputs.last_hidden_state[:, -1, :]

    # Chuyển kết quả về dạng danh sách số (List float) để dễ lưu trữ
    return embeddings.squeeze().tolist()

# Chạy thử nghiệm
# text_sample = "Google Colab T4 là một cứu cánh tuyệt vời cho dân AI."
# vector = get_embedding(text_sample)

# print(f"Độ dài của vector embedding: {len(vector)}") # Thường là 4096 đối với bản 4B
# print(f"Mẫu 5 phần tử đầu tiên: {vector[:5]}")

config.json: 0.00B [00:00, ?B/s]

configuration.py:   0%|          | 0.00/152 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/perplexity-ai/pplx-embed-context-v1-4b:
- configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/perplexity-ai/pplx-embed-context-v1-4b:
- st_quantize.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/perplexity-ai/pplx-embed-context-v1-4b:
- modeling.py
- st_quantize.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The repository perplexity-ai/pplx-embed-context-v1-4b contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/perplexity-ai/pplx-embed-context-v1-4b .
 You can inspect the repository content at https://hf.co/perplexity-ai/pplx-embed-context-v1-4b.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [31]:
import pandas as pd
from pathlib import Path

def filter_by_quarter_range_to_md(
    df, date_start, date_end, title_text, period_col="period"
):
    """Hàm lọc DataFrame theo khoảng thời gian và trả về chuỗi văn bản định dạng

    Markdown (Gồm tiêu đề và bảng dữ liệu).
    """

    if period_col not in df.columns:
        raise ValueError(
            f"Lỗi: Không tìm thấy cột '{period_col}' trong DataFrame của bạn!"
        )

    df_copy = df.copy()

    # 1. Quy đổi ngày bắt đầu và kết thúc sang định dạng Quý (Period)
    start_quarter = pd.to_datetime(date_start, format="%Y-%m-%d").to_period("Q")
    end_quarter = pd.to_datetime(date_end, format="%Y-%m-%d").to_period("Q")

    # 2. Ép kiểu cột period sang PeriodIndex chuyên dụng
    df_copy["_period_typed"] = pd.PeriodIndex(df_copy[period_col], freq="Q")

    # 3. Lọc dữ liệu theo khoảng Quý vừa tính được
    df_filtered = df_copy[
        (df_copy["_period_typed"] >= start_quarter)
        & (df_copy["_period_typed"] <= end_quarter)
    ].copy()

    # 4. Loại bỏ cột phụ
    df_filtered = df_filtered.drop(columns=["_period_typed"])

    # 5. Chuyển đổi bảng DataFrame thành chuỗi text định dạng Markdown
    # Lưu ý: Bạn cần cài đặt thư viện 'tabulate' (pip install tabulate) để hàm này hoạt động mượt mà
    markdown_table = df_filtered.to_markdown(index=False)

    # 6. Ghép tiêu đề và bảng thành một chuỗi văn bản duy nhất
    markdown_output = f"# {title_text}\n\nThời gian: {date_start} đến {date_end} (YYYY-MM-DD)\n\n{markdown_table}"

    return markdown_output

def markdown_report_from_csv(df, title_text):
    """Hàm tiện ích để đọc dữ liệu từ file CSV, lọc theo khoảng thời gian và trả về Markdown."""
    markdown_table = df.to_markdown(index=False)
    markdown_output = f"# {title_text}\n\n{markdown_table}"
    return markdown_output

def get_report(start_date, end_date):
    report_md = []

    income_statement = pd.read_csv('VNM_income_statement.csv')
    report_md.append(filter_by_quarter_range_to_md(income_statement, start_date, end_date, "Báo cáo tài chính quý"))
    balance_sheet = pd.read_csv('VNM_balance_sheet.csv')
    report_md.append(filter_by_quarter_range_to_md(balance_sheet, start_date, end_date, "Bảng cân đối kế toán quý"))

    cash_flow = pd.read_csv('VNM_cash_flow.csv')
    report_md.append(filter_by_quarter_range_to_md(cash_flow, start_date, end_date, "Báo cáo lưu chuyển tiền tệ quý"))

    ratios = pd.read_csv('VNM_ratios.csv')
    report_md.append(filter_by_quarter_range_to_md(ratios, start_date, end_date, "Các chỉ số tài chính quý"))

    # notes = pd.read_csv('VNM_notes.csv')
    # report_md.append(filter_by_quarter_range_to_md(notes, start_date, end_date, "Thuyết minh báo cáo tài chính"))

    financial_health = pd.read_csv('VNM_financial_health.csv')
    report_md.append(filter_by_quarter_range_to_md(financial_health, start_date, end_date, "Đánh giá sức khỏe tài chính quý"))

    info = pd.read_csv('VNM_info.csv')
    report_md.insert(0, markdown_report_from_csv(info, "Thông tin công ty"))

    shareholders = pd.read_csv('VNM_shareholders.csv')
    report_md.insert(1, markdown_report_from_csv(shareholders, "Danh sách cổ đông lớn"))

    news = pd.read_csv('VNM_news.csv')
    filtered_news = news[news['public_date'].between(
        start_date,
        end_date
    )]
    report_md.append(markdown_report_from_csv(filtered_news, "Tin tức liên quan đến công ty"))

    output_md = "\n\n---\n\n".join(report_md)
    output_path = Path(f"financial_report_{start_date}-{end_date}.md")
    output_path.write_text(output_md, encoding="utf-8")
    return output_md

start_date = "2024-04-12"
end_date = "2026-10-10"
output_md = get_report(start_date, end_date)

In [ ]:
import requests
from IPython.display import Markdown, display

API_KEY = "sk-be8bac95ac95e07b38bb3ccaf3e9609875dd1f0c1f537f58eab8490f89d7f619"
model_id = "gpt-5.5"

def chat_with_llm(prompt: str, model_id, API_KEY) -> dict:
    url = "https://ckey.vn/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    data = {
        "model": model_id,
        "messages": [{"role": "user", "content": prompt}],
    }

    response = requests.post(url, headers=headers, json=data)

    return response.json()

USER_PROMPT = """
Bạn là một hệ thống phân tích định lượng tài chính cao cấp (Financial Information Extractor). Hãy đọc tài liệu báo cáo tài chính được cung cấp của công ty VNM (Vinamilk) và trích xuất thông tin thành một bản tóm tắt cấu trúc, cô đọng, tuyệt đối không viết văn dài dòng.

Nhiệm vụ của bạn là trả về kết quả chính xác theo các nhóm chỉ tiêu sau:

1. CHỈ SỐ KHẢ NĂNG SINH LỜI (Profitability):
- Tỷ suất lợi nhuận gộp (Gross Profit Margin), ROA, ROE.
- Nhận diện xu hướng (TĂNG/GIẢM/ỔN ĐỊNH) so với kỳ trước.

2. CHỈ SỐ THANH KHOẢN (Liquidity):
- Tỷ số thanh toán ngắn hạn (Current Ratio), Tỷ số thanh toán nhanh (Quick Ratio).
- Đánh giá trạng thái bằng 1 từ duy nhất: [AN TOÀN] hoặc [RỦI RO].

3. RỦI RO TÀI CHÍNH ĐÒN BẨY (Financial Leverage & Risk):
- Tỷ lệ Nợ/Vốn chủ sở hữu (Debt-to-Equity).
- Tốc độ vòng quay khoản phải thu (Receivables Turnover).

4. KẾT LUẬN ĐỊNH LƯỢNG (Dưới 80 từ):
- Tổng hợp sức khỏe tài chính hiện tại của VNM có thể hỗ trợ chịu đựng các biến cố thị trường ngắn hạn ở mức độ nào. Sử dụng các từ khóa nhãn trạng thái như: [TĂNG TRƯỞNG MẠNH] / [ỔN ĐỊNH PHÒNG THỦ] / [SUY GIẢM] / [RỦI RO CAO].
- Phản hồi bằng tiếng Việt.

Yêu cầu nghiêm ngặt: Chỉ sử dụng các con số thực tế và thuật ngữ tài chính cốt lõi. Loại bỏ toàn bộ các từ nối biểu cảm, dẫn dắt rườm rà để đảm bảo vector embedding sau đó đạt mật độ thông tin (information density) cao nhất, không sử dụng icon hoặc emoji trong bài báo cáo.
"""

try:
    result = chat_with_llm(USER_PROMPT + "\n\n" + output_md, model_id, API_KEY)
    print(result)
    llm_answer = result["choices"][0]["message"]["content"]
    answer_path = Path(f"answer_{model_id}_{start_date}-{end_date}.txt")
    answer_path.write_text(llm_answer, encoding="utf-8")
    display(Markdown(llm_answer))
except Exception as e:
    print(f"Lỗi khi gọi API: {e}")

{'id': 'resp_026ffb5c1c36f795016a11a756eab08196b1a8ee0bbf39705d', 'object': 'chat.completion', 'created': 1779541846, 'model': 'gpt-5.5', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Kỳ phân tích: 2026-Q1  \nKỳ so sánh: 2025-Q4\n\n1. CHỈ SỐ KHẢ NĂNG SINH LỜI\n\n| Chỉ tiêu | 2026-Q1 | 2025-Q4 | Xu hướng |\n|---|---:|---:|---|\n| Gross Profit Margin | 42,70% | 40,45% | TĂNG |\n| ROA | 18,09% | 20,75% | GIẢM |\n| ROE | 27,63% | 31,23% | GIẢM |\n\n2. CHỈ SỐ THANH KHOẢN\n\n| Chỉ tiêu | 2026-Q1 | 2025-Q4 | Xu hướng |\n|---|---:|---:|---|\n| Current Ratio | 2,10x | 1,96x | TĂNG |\n| Quick Ratio | 1,70x | 1,59x | TĂNG |\n\nTrạng thái: [AN TOÀN]\n\n3. RỦI RO TÀI CHÍNH ĐÒN BẨY\n\n| Chỉ tiêu | 2026-Q1 | 2025-Q4 | Xu hướng |\n|---|---:|---:|---|\n| Debt-to-Equity | 0,51x | 0,55x | GIẢM |\n| Receivables Turnover | 11,12x | 11,41x | GIẢM |\n\n4. KẾT LUẬN ĐỊNH LƯỢNG\n\n[ỔN ĐỊNH PHÒNG THỦ] VNM duy trì thanh khoản cao: Current Ratio 2,10x, Quick Ratio 1,70x. Đòn bẩy thấp: Debt-

Kỳ phân tích: 2026-Q1  
Kỳ so sánh: 2025-Q4

1. CHỈ SỐ KHẢ NĂNG SINH LỜI

| Chỉ tiêu | 2026-Q1 | 2025-Q4 | Xu hướng |
|---|---:|---:|---|
| Gross Profit Margin | 42,70% | 40,45% | TĂNG |
| ROA | 18,09% | 20,75% | GIẢM |
| ROE | 27,63% | 31,23% | GIẢM |

2. CHỈ SỐ THANH KHOẢN

| Chỉ tiêu | 2026-Q1 | 2025-Q4 | Xu hướng |
|---|---:|---:|---|
| Current Ratio | 2,10x | 1,96x | TĂNG |
| Quick Ratio | 1,70x | 1,59x | TĂNG |

Trạng thái: [AN TOÀN]

3. RỦI RO TÀI CHÍNH ĐÒN BẨY

| Chỉ tiêu | 2026-Q1 | 2025-Q4 | Xu hướng |
|---|---:|---:|---|
| Debt-to-Equity | 0,51x | 0,55x | GIẢM |
| Receivables Turnover | 11,12x | 11,41x | GIẢM |

4. KẾT LUẬN ĐỊNH LƯỢNG

[ỔN ĐỊNH PHÒNG THỦ] VNM duy trì thanh khoản cao: Current Ratio 2,10x, Quick Ratio 1,70x. Đòn bẩy thấp: Debt-to-Equity 0,51x. Biên gộp tăng 42,70%. ROA 18,09%, ROE 27,63% giảm QoQ nhưng vẫn cao. Khả năng chịu biến cố thị trường ngắn hạn: tốt.

In [ ]:
# Cấu hình thông tin
result = get_embedding(llm_answer)
print(f"Độ dài của vector embedding: {len(result)}")
print(result)

Độ dài của vector embedding: 2560
[-0.0007472038269042969, 0.1700439453125, -0.231201171875, 0.326171875, -0.0024929046630859375, -0.038482666015625, 0.463623046875, 1.3642578125, 0.464111328125, -0.98193359375, 0.2076416015625, -0.108642578125, -0.5244140625, -1.2060546875, 1.0615234375, -0.397705078125, 0.69580078125, 0.234130859375, 2.330078125, -0.255615234375, -0.2403564453125, 1.2119140625, 0.388671875, -0.19921875, -1.7109375, 0.7451171875, -0.4521484375, 3.23046875, 0.221923828125, -0.437744140625, -0.8173828125, -0.345458984375, -0.29833984375, 0.292236328125, 0.2442626953125, 0.77001953125, -0.274169921875, -0.42041015625, -0.1590576171875, 0.1357421875, 0.1402587890625, -0.529296875, 0.2200927734375, 0.2445068359375, -0.73876953125, 0.417236328125, 0.1455078125, 0.974609375, -0.1717529296875, -0.405029296875, -0.273681640625, 0.7626953125, 0.362060546875, -0.03448486328125, 0.04168701171875, -0.6884765625, 2.8984375, 0.400390625, -0.25830078125, 0.97216796875, -0.5361328125,

In [33]:
import pandas as pd
from pathlib import Path
import requests
import gradio as gr

# --- PHẦN 1: CÁC HÀM XỬ LÝ DỮ LIỆU BÁO CÁO ---

def filter_by_quarter_range_to_md(df, date_start, date_end, title_text, period_col="period"):
    """Hàm lọc DataFrame theo khoảng thời gian và trả về chuỗi văn bản định dạng Markdown."""
    if period_col not in df.columns:
        raise ValueError(f"Lỗi: Không tìm thấy cột '{period_col}' trong DataFrame của bạn!")

    df_copy = df.copy()
    start_quarter = pd.to_datetime(date_start, format="%Y-%m-%d").to_period("Q")
    end_quarter = pd.to_datetime(date_end, format="%Y-%m-%d").to_period("Q")

    df_copy["_period_typed"] = pd.PeriodIndex(df_copy[period_col], freq="Q")
    df_filtered = df_copy[
        (df_copy["_period_typed"] >= start_quarter)
        & (df_copy["_period_typed"] <= end_quarter)
    ].copy()

    df_filtered = df_filtered.drop(columns=["_period_typed"])
    markdown_table = df_filtered.to_markdown(index=False)
    markdown_output = f"# {title_text}\n\nThời gian: {date_start} đến {date_end} (YYYY-MM-DD)\n\n{markdown_table}"

    return markdown_output

def markdown_report_from_csv(df, title_text):
    """Hàm tiện ích để đọc dữ liệu từ file CSV, lọc theo khoảng thời gian và trả về Markdown."""
    markdown_table = df.to_markdown(index=False)
    markdown_output = f"# {title_text}\n\n{markdown_table}"
    return markdown_output

def get_report(start_date, end_date):
    """Hàm tổng hợp các file CSV thành một báo cáo Markdown hoàn chỉnh."""
    report_md = []

    income_statement = pd.read_csv('VNM_income_statement.csv')
    report_md.append(filter_by_quarter_range_to_md(income_statement, start_date, end_date, "Báo cáo tài chính quý"))

    balance_sheet = pd.read_csv('VNM_balance_sheet.csv')
    report_md.append(filter_by_quarter_range_to_md(balance_sheet, start_date, end_date, "Bảng cân đối kế toán quý"))

    cash_flow = pd.read_csv('VNM_cash_flow.csv')
    report_md.append(filter_by_quarter_range_to_md(cash_flow, start_date, end_date, "Báo cáo lưu chuyển tiền tệ quý"))

    ratios = pd.read_csv('VNM_ratios.csv')
    report_md.append(filter_by_quarter_range_to_md(ratios, start_date, end_date, "Các chỉ số tài chính quý"))

    financial_health = pd.read_csv('VNM_financial_health.csv')
    report_md.append(filter_by_quarter_range_to_md(financial_health, start_date, end_date, "Đánh giá sức khỏe tài chính quý"))

    info = pd.read_csv('VNM_info.csv')
    report_md.insert(0, markdown_report_from_csv(info, "Thông tin công ty"))

    shareholders = pd.read_csv('VNM_shareholders.csv')
    report_md.insert(1, markdown_report_from_csv(shareholders, "Danh sách cổ đông lớn"))

    news = pd.read_csv('VNM_news.csv')
    filtered_news = news[news['public_date'].between(start_date, end_date)]
    report_md.append(markdown_report_from_csv(filtered_news, "Tin tức liên quan đến công ty"))

    output_md = "\n\n---\n\n".join(report_md)
    output_path = Path(f"financial_report_{start_date}-{end_date}.md")
    output_path.write_text(output_md, encoding="utf-8")

    return output_md

# --- PHẦN 2: HÀM GỌI API VÀ ĐIỀU PHỐI CHUNG ---

def chat_with_llm(prompt: str, model_id: str, api_key: str) -> dict:
    url = "https://ckey.vn/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    data = {
        "model": model_id,
        "messages": [{"role": "user", "content": prompt}],
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()

def process_financial_data(date_start, date_end, user_prompt, model_id, api_key):
    """Hàm động cơ chính: Nhận dữ liệu từ UI, chạy logic và trả về kết quả cho UI."""
    # Bước 1: Lấy báo cáo định dạng Markdown
    try:
        output_md = get_report(date_start, date_end)
    except Exception as e:
        return f"Lỗi khi đọc file hoặc tạo báo cáo: {e}", "Không có dữ liệu báo cáo để gửi đi."

    # Bước 2: Ghép nối dữ liệu và gọi API
    try:
        full_prompt = user_prompt + "\n\n" + output_md
        result = chat_with_llm(full_prompt, model_id, api_key)

        # Kiểm tra và trích xuất câu trả lời
        if "choices" in result and len(result["choices"]) > 0:
            llm_answer = result["choices"][0]["message"]["content"]

            # Lưu file text kết quả
            answer_path = Path(f"answer_{model_id}_{date_start}-{date_end}.txt")
            answer_path.write_text(llm_answer, encoding="utf-8")

            return output_md, llm_answer
        else:
            error_msg = f"Lỗi từ hệ thống API: {result}"
            return output_md, error_msg

    except Exception as e:
        return output_md, f"Lỗi hệ thống khi kết nối API: {e}"

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://555ad94e8026feca4b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
DEFAULT_PROMPT = """Bạn là một hệ thống phân tích định lượng tài chính cao cấp (Financial Information Extractor). Hãy đọc tài liệu báo cáo tài chính được cung cấp của công ty VNM (Vinamilk) và trích xuất thông tin thành một bản tóm tắt cấu trúc, cô đọng, tuyệt đối không viết văn dài dòng.

Nhiệm vụ của bạn là trả về kết quả chính xác theo các nhóm chỉ tiêu sau:

1. CHỈ SỐ KHẢ NĂNG SINH LỜI (Profitability):
- Tỷ suất lợi nhuận gộp (Gross Profit Margin), ROA, ROE.
- Nhận diện xu hướng (TĂNG/GIẢM/ỔN ĐỊNH) so với kỳ trước.

2. CHỈ SỐ THANH KHOẢN (Liquidity):
- Tỷ số thanh toán ngắn hạn (Current Ratio), Tỷ số thanh toán nhanh (Quick Ratio).
- Đánh giá trạng thái bằng 1 từ duy nhất: [AN TOÀN] hoặc [RỦI RO].

3. RỦI RO TÀI CHÍNH ĐÒN BẨY (Financial Leverage & Risk):
- Tỷ lệ Nợ/Vốn chủ sở hữu (Debt-to-Equity).
- Tốc độ vòng quay khoản phải thu (Receivables Turnover).

4. KẾT LUẬN ĐỊNH LƯỢNG (Dưới 80 từ):
- Tổng hợp sức khỏe tài chính hiện tại của VNM có thể hỗ trợ chịu đựng các biến cố thị trường ngắn hạn ở mức độ nào. Sử dụng các từ khóa nhãn trạng thái như: [TĂNG TRƯỞNG MẠNH] / [ỔN ĐỊNH PHÒNG THỦ] / [SUY GIẢM] / [RỦI RO CAO].
- Phản hồi bằng tiếng Việt.

Yêu cầu nghiêm ngặt: Chỉ sử dụng các con số thực tế và thuật ngữ tài chính cốt lõi. Loại bỏ toàn bộ các từ nối biểu cảm, dẫnạc rườm rà để đảm bảo vector embedding sau đó đạt mật độ thông tin (information density) cao nhất, không sử dụng icon hoặc emoji trong bài báo cáo."""

with gr.Blocks(title="Hệ Thống Phân Tích Tài Chính") as app:
    gr.Markdown("# Bảng Điều Khiển Phân Tích Định Lượng VNM")

    with gr.Row():
        # Cột bên trái: Tiếp nhận thông tin (Inputs)
        with gr.Column(scale=1):
            gr.Markdown("### Thông số cấu hình")
            start_date_input = gr.Textbox(label="Ngày bắt đầu (YYYY-MM-DD)", value="2024-04-12")
            end_date_input = gr.Textbox(label="Ngày kết thúc (YYYY-MM-DD)", value="2026-10-10")
            model_id_input = gr.Textbox(label="Tên Model", value="gpt-5.5")

            # Sử dụng type="password" để ẩn các ký tự API Key khi nhập vào
            api_key_input = gr.Textbox(label="API Key", type="password", placeholder="Nhập API Key sk-...")

            prompt_input = gr.Textbox(
                label="Câu lệnh hệ thống (System Prompt)",
                lines=15,
                value=DEFAULT_PROMPT
            )
            submit_btn = gr.Button("Trích xuất và Phân tích", variant="primary")

        # Cột bên phải: Hiển thị kết quả AI (Outputs)
        with gr.Column(scale=1):
            gr.Markdown("### Phản hồi từ Hệ thống AI")
            llm_output_display = gr.Markdown(label="Kết quả phân tích")

    # Hàng ngang phía dưới: Chứa nội dung báo cáo thô có thể đóng mở
    with gr.Row():
        with gr.Accordion("Xem dữ liệu báo cáo thô (Markdown)", open=False):
            report_output_display = gr.Markdown()

    # Kết nối nút bấm với động cơ xử lý trung tâm
    submit_btn.click(
        fn=process_financial_data,
        inputs=[start_date_input, end_date_input, prompt_input, model_id_input, api_key_input],
        outputs=[report_output_display, llm_output_display]
    )

# Khởi chạy ứng dụng
if __name__ == "__main__":
    app.launch()